<a href="https://colab.research.google.com/github/Na-bra/recommendation-agent/blob/main/Recommender_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install surprise

In [2]:
!pip3 install scikit-surprise
!pip3 install --force-reinstall -v "numpy<2.0.0"

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/3a/be/650f9c091ef71cb01d735775d554e068752d3ff63d7943b26316dc401749/numpy-1.21.2.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/5f/d6/ad58ded26556eaeaa8c971e08b6466f17c4ac4d786cd3d800e26ce59cc01/numpy-1.21.3.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/fb/48/b0708ebd7718a8933f0d3937513ef8ef2f4f04529f1f66ca86d873043921/numpy-1.21.4.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.13 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/c2/a8/a924a09492bdfee8c2ec3094d0

Imports

In [1]:
from surprise import Dataset,Reader
from surprise.prediction_algorithms import SVD
from surprise import accuracy
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

Load Data and Preprocess

In [3]:
path: str = "/content/drive/MyDrive/movie-data"
ratings_df = pd.read_csv(f"{path}/ratings.csv")
movies_df = pd.read_csv(f"{path}/movies.csv")

df = pd.merge(ratings_df,movies_df[['movieId', 'genres']], on = 'movieId', how = 'left')

Clean Data

In [4]:
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()
mlb = MultiLabelBinarizer()
df['userId'] = user_encoder.fit_transform(df['userId'])
df['movieId'] = user_encoder.fit_transform(df['movieId'])
df = df.join(pd.DataFrame(mlb.fit_transform(df.pop('genres').str.split('|')),columns = mlb.classes_, index = df.index))

In [5]:
df.drop(columns = "(no genres listed)", inplace = True)
df

,userId,movieId,rating,timestamp,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,4.0,964982703,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,2,4.0,964981247,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
2,0,5,4.0,964982224,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,0,43,5.0,964983815,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,0,46,5.0,964982931,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100831,609,9416,4.0,1493848402,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
100832,609,9443,5.0,1493850091,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
100833,609,9444,5.0,1494273047,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
100834,609,9445,5.0,1493846352,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


Build The Model With Collaborative Filtering

In [6]:
train_df, test_df = train_test_split(df, test_size = 0.2)
train_df

,userId,movieId,rating,timestamp,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
60968,394,97,4.0,841503580,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
49774,317,8408,3.0,1415452491,1,1,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
3368,20,3827,4.0,1403459855,1,1,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
28989,199,913,4.0,1229886360,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
87151,561,1471,3.5,1368892691,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83364,527,2370,3.5,1391736303,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
16894,104,9196,5.0,1526207987,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
37819,255,5213,4.5,1446580635,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
96368,602,897,3.0,953927628,1,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [7]:
reader = Reader(rating_scale = (0.5, 5.0))
data = Dataset.load_from_df(train_df[['userId','movieId','rating']], reader)
trainset = data.build_full_trainset()
trainset

In [8]:
model_svd = SVD()
model_svd.fit(trainset)
predictions_svd = model_svd.test(trainset.build_anti_testset())
accuracy.rmse(predictions_svd)

RMSE: 0.4743


0.4743353871242864

Make Recommendations

In [ ]:
def get_top_n_reconmmendations(user_id, n = 5):
  user_movies = df[df['userId'] == user_id]['movieId'].unique()
  all_movies = df['movies'].unique()
  movies_to_predict = list(set(all_movies) - set(user_movies))
  user